# 8-6(참고). σ 전파·드리프트 검증 (GARCH·SV 구현 차이)

결함판 `8-5`에서 세 모델의 구현 차이 두 가지를 발견했다(결함판은 `결함판(참고용)/`). 이 차이가 결함판 `8-2`~`8-4`의 결론에 영향을 주는지 d=1, p=1에서 확인한다.

1. **σ 전파 오류**: AR단독은 `pmdarima`의 정확한 다단계 σ를 쓰지만, GARCH·SV는 단계별 분산의 누적합(`propagate_var`)을 써서 h=2~4의 σ를 과소 계산한다.
2. **드리프트 비대칭(비교 조건 불일치)**: AR단독은 드리프트가 없고, GARCH는 상수항(`arch` 기본값), SV는 훈련 평균을 더한다. 계산 오류는 아니지만 비교 조건이 달라 통일해서 영향을 본다.

방법: 설정만 바꿔 같은 walk-forward(22개 origin x h=1~4)로 커버리지·MSE·CRPS와 점예측 차이를 비교한다.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import pmdarima as pm
import properscoring as ps
from arch.univariate import ARX, GARCH
from scipy import stats
from scipy.optimize import minimize

H, MIN_TRAIN, INITIAL_CALIB, ALPHA = 4, 20, 15, 0.10
Z = stats.norm.ppf(1 - ALPHA / 2)
XI_MEAN, XI_VAR, MAX_PHI = -1.2704, np.pi ** 2 / 2, 0.97

raw = pd.read_csv("../../data/gold/gold_major_variables.csv")
raw = raw[raw["quarter"].notna()].reset_index(drop=True)
vals = raw[["quarter", "X3_l1"]].dropna()["X3_l1"].values.astype(float)
var_std = vals.std()
scored_origins = list(range(MIN_TRAIN, len(vals) - H + 1))[INITIAL_CALIB:]
print(f"채점 origin {len(scored_origins)}개, d=1, p=1 고정")


채점 origin 22개, d=1, p=1 고정


## 1. 세 모델을 설정만 바꿔 돌리는 함수

- `level_var`: 차분이 AR(1)일 때 h단계 **수준** 오차의 정확한 분산. 수준 오차는 `Σ ε_(t+i) x (1+φ+…+φ^(h-i))`로 쌓이므로 `Σ_i (1+φ+…+φ^(h-i))² x (i단계 오차분산)`이다(누적합은 이 계수를 1로 둔 것).
- `drift`: AR단독은 `with_intercept`, GARCH는 상수항, SV는 평균 제거·복원 여부.
- `sigma_fix`: GARCH·SV의 σ를 누적합(현재) 대신 `level_var`(정확)로 계산할지.


In [2]:
def sv_qml_negloglik(z_centered, phi, sigma_eta2):
    P0 = sigma_eta2 / (1 - phi ** 2) if abs(phi) < 0.999 else sigma_eta2 * 100
    h_pred, P_pred, loglik = 0.0, P0, 0.0
    for z in z_centered:
        F = P_pred + XI_VAR
        v = z - h_pred
        loglik += -0.5 * (np.log(2 * np.pi * F) + v ** 2 / F)
        K = P_pred / F
        h_filt, P_filt = h_pred + K * v, P_pred * (1 - K)
        h_pred, P_pred = phi * h_filt, phi ** 2 * P_filt + sigma_eta2
    return -loglik


def joint_nll(params, y):
    ar, phi, s2 = params[0], MAX_PHI * np.tanh(params[1]), np.exp(params[2]) ** 2
    if abs(ar) >= 0.999:
        return 1e10
    z = np.log((y[1:] - ar * y[:-1]) ** 2 + 1e-8)
    return sv_qml_negloglik(z - z.mean(), phi, s2)


def level_var(phi, v):
    return np.array([sum(sum(phi ** j for j in range(h - i + 1)) ** 2 * v[i - 1] for i in range(1, h + 1)) for h in range(1, H + 1)])


def ar_forecast(train, drift, sigma_fix):
    m = pm.ARIMA(order=(1, 1, 0), with_intercept=drift, suppress_warnings=True).fit(train)
    fc, conf = m.predict(n_periods=H, return_conf_int=True, alpha=ALPHA)
    ar = dict(zip(m.arima_res_.model.param_names, m.arima_res_.params))["ar.L1"]
    return np.asarray(fc), (conf[:, 1] - conf[:, 0]) / (2 * Z), ar


def garch_forecast(train, drift, sigma_fix):
    am = ARX(np.diff(train), lags=1, constant=drift, rescale=False)
    am.volatility = GARCH(p=1, q=1)
    res = am.fit(disp="off")
    f = res.forecast(horizon=H, reindex=False)
    phi = res.params["y[1]"]
    var = level_var(phi, f.residual_variance.values[-1]) if sigma_fix else np.cumsum(f.variance.values[-1])
    return train[-1] + np.cumsum(f.mean.values[-1]), np.sqrt(var), phi


def sv_forecast(train, drift, sigma_fix):
    y = np.diff(train)
    ym = y.mean() if drift else 0.0
    yd = y - ym
    res = minimize(joint_nll, [0.0, 1.5, np.log(0.3)], args=(yd,), method="Nelder-Mead", options={"maxiter": 3000})
    ar, phi, s2 = res.x[0], MAX_PHI * np.tanh(res.x[1]), np.exp(res.x[2]) ** 2
    z = np.log((yd[1:] - ar * yd[:-1]) ** 2 + 1e-8)
    zm = z.mean()
    P, h = s2 / (1 - phi ** 2), 0.0
    for x in z - zm:
        K = P / (P + XI_VAR)
        hf, Pf = h + K * (x - h), P * (1 - K)
        h, P = phi * hf, phi ** 2 * Pf + s2
    mean_dm = np.array([ar ** k * yd[-1] for k in range(1, H + 1)])
    v = np.array([np.exp(zm - XI_MEAN + phi ** k * hf + (phi ** (2 * k) * Pf + s2 * sum(phi ** (2 * j) for j in range(k))) / 2)
                  for k in range(1, H + 1)])
    var = level_var(ar, v) if sigma_fix else np.cumsum(v)
    return train[-1] + np.cumsum(mean_dm + ym), np.sqrt(var), ar


MODELS = {"AR단독": ar_forecast, "GARCH동시": garch_forecast, "SV동시": sv_forecast}
NO, YES = {m: False for m in MODELS}, {m: True for m in MODELS}
CURRENT = {"AR단독": False, "GARCH동시": True, "SV동시": True}   # 결함판 8-4의 설정
CONFIGS = {
    "초기 설정": (CURRENT, False),
    "σ만 수정": (CURRENT, True),
    "드리프트 없음 통일": (NO, False),
    "드리프트 있음 통일": (YES, False),
    "σ 수정 + 드리프트 없음": (NO, True),
    "σ 수정 + 드리프트 있음": (YES, True),
}


def run(drift_by_model, sigma_fix):
    rows = []
    for origin in scored_origins:
        train, actual = vals[:origin], vals[origin:origin + H]
        for name, fn in MODELS.items():
            fc, sig, ar = fn(train, drift_by_model[name], sigma_fix)
            for h in range(H):
                rows.append({"모델": name, "origin": origin, "h": h + 1, "fc": fc[h], "sigma": max(sig[h], 1e-6),
                             "actual": actual[h], "ar": ar})
    df = pd.DataFrame(rows)
    df["z"] = (df["actual"] - df["fc"]) / df["sigma"]
    df["cov"] = (df["z"].abs() <= Z) * 100.0
    df["mse"] = ((df["actual"] - df["fc"]) / var_std) ** 2
    df["crps"] = ps.crps_gaussian(df["actual"], mu=df["fc"], sig=df["sigma"]) / var_std
    return df


results = {name: run(*cfg) for name, cfg in CONFIGS.items()}
print("설정", len(results), "개 계산 완료")


설정 6 개 계산 완료


## 2. 재현 확인 -- "초기 설정" 설정이 결함판 `8-4`의 d=1, p=1 결과와 같은가

새로 짠 함수가 기존 파이프라인을 그대로 재현해야 이후 비교가 의미가 있다.


In [3]:
def summarize(df):
    return df.groupby("모델").agg(커버리지=("cov", "mean"), MSE=("mse", "mean"), CRPS=("crps", "mean")).round(3)


orig = pd.read_pickle("../cache/cache_84_dp_grid_result_df.pkl")
orig = orig[(orig["d"] == 1) & (orig["p"] == 1)].groupby("모델").agg(커버리지=("cov", "mean"), MSE=("mse", "mean"), CRPS=("crps", "mean")).round(3)
check = summarize(results["초기 설정"]).join(orig, lsuffix="_재계산", rsuffix="_결함판8-4")
display(check)
print("최대 차이:", float((summarize(results["초기 설정"]) - orig).abs().max().max()))


,커버리지_재계산,MSE_재계산,CRPS_재계산,커버리지_결함판8-4,MSE_결함판8-4,CRPS_결함판8-4
모델,,,,,,
AR단독,65.909,1.128,0.593,65.909,1.128,0.593
GARCH동시,65.909,1.114,0.604,65.909,1.114,0.604
SV동시,69.318,1.086,0.591,69.318,1.086,0.591


최대 차이: 0.0


## 3. σ 전파 수정의 효과 (드리프트는 현재 설정 그대로)

GARCH·SV의 σ만 정확한 공식으로 바꾼다. AR단독은 원래 정확해서 그대로다. h별 z 표준편차(1에 가까울수록 σ가 맞음)와 성능을 본다.


In [4]:
zstd = pd.concat({k: results[k].groupby(["모델", "h"])["z"].std().unstack("h") for k in ["초기 설정", "σ만 수정"]}, names=["설정"]).round(2)
display(zstd)
display(pd.concat({k: summarize(results[k]) for k in ["초기 설정", "σ만 수정"]}, names=["설정"]))


h                 1     2     3     4
설정    모델                             
초기 설정 AR단독     2.11  2.19  2.18  2.11
      GARCH동시  2.07  2.51  2.65  2.68
      SV동시     1.52  1.85  1.97  2.03
σ만 수정 AR단독     2.11  2.19  2.18  2.11
      GARCH동시  2.07  2.15  2.09  2.00
      SV동시     1.52  1.87  2.01  2.06

커버리지    MSE   CRPS
설정    모델                           
초기 설정 AR단독     65.909  1.128  0.593
      GARCH동시  65.909  1.114  0.604
      SV동시     69.318  1.086  0.591
σ만 수정 AR단독     65.909  1.128  0.593
      GARCH동시  71.591  1.114  0.595
      SV동시     69.318  1.086  0.591

**결과**

- GARCH동시의 h별 z 표준편차가 `2.07 -> 2.51 -> 2.65 -> 2.68`에서 `2.07 -> 2.15 -> 2.09 -> 2.00`으로 **평탄해진다**. 누적합의 과소 계산이 h가 커질수록 z가 커지던 원인이었다. 커버리지 65.9 -> 71.6%, CRPS 0.604 -> 0.595.
- SV동시는 거의 그대로다(AR계수≈0이라 누적합이 거의 정확). SV의 h별 증가(1.52 -> 2.03)는 다른 원인이며 미확인이다.
- 수정 후에도 세 모델의 z 표준편차가 1.5~2.2라서 σ 과소라는 근본 문제는 남는다.


## 4. 드리프트 통일의 효과 (σ는 현재 방식 그대로)

세 모델의 드리프트 처리를 통일한다 -- 모두 없음 / 모두 있음. 점예측이 모델마다 달라 보이는 정도(결함판 `8-2` 3절과 같은 지표: 두 모델 점예측 차이의 평균 / `var_std`)와 AR계수, 성능이 어떻게 바뀌는지 본다.


In [5]:
def fc_gap(df):
    w = df.pivot_table(index=["origin", "h"], columns="모델", values="fc")
    return pd.Series({"|AR-GARCH|": (w["AR단독"] - w["GARCH동시"]).abs().mean() / var_std,
                      "|AR-SV|": (w["AR단독"] - w["SV동시"]).abs().mean() / var_std,
                      "|GARCH-SV|": (w["GARCH동시"] - w["SV동시"]).abs().mean() / var_std})


drift_keys = ["초기 설정", "드리프트 없음 통일", "드리프트 있음 통일"]
display(pd.DataFrame({k: fc_gap(results[k]) for k in drift_keys}).T.round(3).rename_axis("설정"))
print("모델별 AR계수 평균:")
display(pd.DataFrame({k: results[k].groupby("모델")["ar"].mean() for k in drift_keys}).T.round(3).rename_axis("설정"))
display(pd.concat({k: summarize(results[k]) for k in drift_keys}, names=["설정"]))


,|AR-GARCH|,|AR-SV|,|GARCH-SV|
설정,,,
초기 설정,0.087,0.316,0.250
드리프트 없음 통일,0.064,0.315,0.254
드리프트 있음 통일,0.106,0.283,0.250


모델별 AR계수 평균:


모델,AR단독,GARCH동시,SV동시
설정,,,
초기 설정,0.492,0.424,0.010
드리프트 없음 통일,0.492,0.425,-0.056
드리프트 있음 통일,0.474,0.424,0.010


커버리지    MSE   CRPS
설정         모델                           
초기 설정      AR단독     65.909  1.128  0.593
           GARCH동시  65.909  1.114  0.604
           SV동시     69.318  1.086  0.591
드리프트 없음 통일 AR단독     65.909  1.128  0.593
           GARCH동시  70.455  1.080  0.588
           SV동시     65.909  1.052  0.576
드리프트 있음 통일 AR단독     62.500  1.171  0.618
           GARCH동시  65.909  1.114  0.604
           SV동시     69.318  1.086  0.591

**결과**

- **점예측 차이는 드리프트 때문이 아니다**: |AR-SV|가 0.316 -> 0.315(없음)/0.283(있음)로 거의 그대로이고 SV의 AR계수도 0 근처다. 원인은 AR계수 차이다.
- |AR-GARCH|는 0.087 -> 0.064(없음 통일)로 줄어 드리프트가 그 차이의 약 1/4를 설명한다.
- **이 표본에서는 드리프트가 없는 쪽이 낫다**: 없음 통일에서 GARCH(CRPS 0.604 -> 0.588)·SV(0.591 -> 0.576)가 좋아지고, 있음 통일에서는 AR단독이 나빠진다(CRPS 0.593 -> 0.618).

## 5. 둘 다 통일했을 때

σ를 정확하게 하고 드리프트도 통일한 설정에서 세 모델의 비교가 어떻게 되는지 본다.


In [6]:
both_keys = ["초기 설정", "σ 수정 + 드리프트 없음", "σ 수정 + 드리프트 있음"]
display(pd.concat({k: summarize(results[k]) for k in both_keys}, names=["설정"]))
display(pd.DataFrame({k: fc_gap(results[k]) for k in both_keys}).T.round(3).rename_axis("설정"))


커버리지    MSE   CRPS
설정             모델                           
초기 설정          AR단독     65.909  1.128  0.593
               GARCH동시  65.909  1.114  0.604
               SV동시     69.318  1.086  0.591
σ 수정 + 드리프트 없음 AR단독     65.909  1.128  0.593
               GARCH동시  73.864  1.080  0.582
               SV동시     64.773  1.052  0.578
σ 수정 + 드리프트 있음 AR단독     62.500  1.171  0.618
               GARCH동시  71.591  1.114  0.595
               SV동시     69.318  1.086  0.591

,|AR-GARCH|,|AR-SV|,|GARCH-SV|
설정,,,
초기 설정,0.087,0.316,0.250
σ 수정 + 드리프트 없음,0.064,0.315,0.254
σ 수정 + 드리프트 있음,0.106,0.283,0.250


## 6. 결론

d=1, p=1에서 초기 설정과 "σ 수정 + 드리프트 없음" 설정의 비교:

| 모델 | 커버리지(%) | MSE | CRPS |
|---|---|---|---|
| AR단독 | 65.9 -> 65.9 | 1.128 -> 1.128 | 0.593 -> 0.593 |
| GARCH동시 | 65.9 -> 73.9 | 1.114 -> 1.080 | 0.604 -> 0.582 |
| SV동시 | 69.3 -> 64.8 | 1.086 -> 1.052 | 0.591 -> 0.578 |

- **σ 전파 오류는 GARCH동시를 불리하게 만들었다**: 고치면 CRPS가 0.604 -> 0.595, 드리프트까지 통일하면 0.582로 최하위에서 2위가 된다.
- **점예측 차이는 드리프트 비대칭 때문이 아니다**(GARCH-AR 차이의 약 1/4만 드리프트 탓).
- **SV동시가 최선이라는 순위는 유지**되지만 차이는 작아졌다(CRPS 0.578~0.593).
- 커버리지는 어느 설정에서도 62.5~73.9%로 목표 90%에 못 미친다(σ 과소는 이 두 항목과 별개).
- 이를 고친 파이프라인이 `8-2`, `8-4`이고 최종 (d,p)=(1,1)은 유지됐다. 유의성은 검증하지 않았다.